Imports & Hardware Setup

In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torchvision.transforms.functional as TF
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import timm
import json
from shapely.geometry import Point, shape
from PIL import Image

# Add scripts folder to path so we can import our new files
from scripts.model_hierarchical import GeoguessrHierarchicalModel
from scripts.datasets_hierarchical import GeoguessrHierarchicalDataset
from scripts.losses_hierarchical import HaversineSmoothedCrossEntropy, GeometricHuberLoss, latlon_to_cartesian, cartesian_to_latlon, haversine_distance

# [SPEED 1+2+3] Full hardware acceleration
torch.backends.cuda.matmul.allow_tf32 = True
cudnn.allow_tf32 = True
cudnn.benchmark = True  # SPEED 2: Auto-tune convolution algorithms
torch.set_float32_matmul_precision('medium')  # SPEED 3: Global TF32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device} | TF32 Enabled: {torch.backends.cuda.matmul.allow_tf32} | cuDNN Benchmark: {cudnn.benchmark}")

# Create submissions directory
os.makedirs('submissions', exist_ok=True)

Using device: cuda | TF32 Enabled: True | cuDNN Benchmark: True


Hyperparameters & Paths

In [3]:
# --- PATHS ---
BASE_DIR = r"C:\Users\Yash T\Desktop\geoguessr\geolocation-prediction"
TRAIN_CSV = os.path.join(BASE_DIR, "training_dataset", "noised_dataset", "hierarchical_training_data.csv")
TRAIN_IMG_DIR = os.path.join(BASE_DIR, "training_dataset", "noised_dataset", "images")

EXTRA_CSV = os.path.join(BASE_DIR, "extra_training_dataset", "extra_hierarchical_clustered.csv")
EXTRA_IMG_DIR = os.path.join(BASE_DIR, "extra_training_dataset", "images")

TEST_IMG_DIR = os.path.join(BASE_DIR, "test_images_sampled")
GEOJSON_PATH = os.path.join(BASE_DIR, "country_boundaries.geojson")

SAMPLE_SUB_PATH = os.path.join(BASE_DIR, "submissions", "sample_submission.csv")

# Precomputed distance matrices from Step 1
DIST_MAT_1 = os.path.join(BASE_DIR, "training_dataset", "noised_dataset", "hierarchical_clusters", "dist_matrix_macro_4.npy")
DIST_MAT_2 = os.path.join(BASE_DIR, "training_dataset", "noised_dataset", "hierarchical_clusters", "dist_matrix_meso_36.npy")
DIST_MAT_3 = os.path.join(BASE_DIR, "training_dataset", "noised_dataset", "hierarchical_clusters", "dist_matrix_micro_216.npy")

# --- HYPERPARAMETERS ---
BATCH_SIZE = 20
NUM_WORKERS = 16
LEARNING_RATE_BACKBONE = 2e-6
LEARNING_RATE_LAYER_1 = 1e-4
LEARNING_RATE_LAYER_2 = 4e-5

# Smoothing Sigmas (km)
SIGMA_1 = 2500.0
SIGMA_2 = 500.0
SIGMA_3 = 225.0

# Huber Loss Threshold
HUBER_THRESHOLD_KM = 500.0

Dynamically Configured Augmentations

In [3]:
# Dynamically get the exact configuration for the chosen backbone
temp_model = timm.create_model('vit_base_patch16_siglip_256', pretrained=False)
data_config = timm.data.resolve_data_config({}, model=temp_model)

MEAN = data_config['mean']
STD = data_config['std']
INPUT_SIZE = data_config['input_size']
IMG_H, IMG_W = INPUT_SIZE[1], INPUT_SIZE[2]

print(f"Backbone Config -> Resolution: {IMG_H}x{IMG_W} | Mean: {MEAN} | Std: {STD}")

train_transform = A.Compose([
    A.RandomResizedCrop(size=(IMG_H, IMG_W), scale=(0.80, 0.95), p=1.0),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02, p=1.0),
    A.GaussianBlur(blur_limit=(2, 6), p=1.0),
    A.ImageCompression(quality_range=(80, 95), p=1.0),
    A.CoarseDropout(num_holes_range=(2, 5), hole_height_range=(1, 10), hole_width_range=(1, 10), p=1.0),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_H, IMG_W),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

del temp_model

Backbone Config -> Resolution: 256x256 | Mean: (0.5, 0.5, 0.5) | Std: (0.5, 0.5, 0.5)


Data Loaders

In [4]:
print("Initializing Original Dataset Loader...")
train_dataset = GeoguessrHierarchicalDataset(
    csv_path=TRAIN_CSV, 
    image_dir=TRAIN_IMG_DIR, 
    transform=train_transform
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS, 
    pin_memory=True, 
    persistent_workers=(NUM_WORKERS > 0)
)
print(f"Loaded {len(train_dataset)} images into training loader.")

Initializing Original Dataset Loader...
Loaded 19002 images into training loader.


Initialize Model, Losses & Data Loader Helpers

In [5]:
import time
import tempfile
# --- INITIALIZE MODEL & LOSSES ---
print("Initializing SigLIP Hierarchical Model...")
model = GeoguessrHierarchicalModel(backbone_name='vit_base_patch16_siglip_256', pretrained=True)
model.to(device)
# Initialize our specialized losses
loss_fn_h1 = HaversineSmoothedCrossEntropy(DIST_MAT_1, sigma=SIGMA_1, device=device)
loss_fn_h2 = HaversineSmoothedCrossEntropy(DIST_MAT_2, sigma=SIGMA_2, device=device)
loss_fn_h3 = HaversineSmoothedCrossEntropy(DIST_MAT_3, sigma=SIGMA_3, device=device)
loss_fn_coord = GeometricHuberLoss(threshold_km=HUBER_THRESHOLD_KM)
# --- DATA LOADER HELPERS ---
def get_extra_loader(folder_id):
    """Dynamically loads a chunk of the extra dataset."""
    df_chunk = pd.read_csv(EXTRA_CSV, dtype={'folder': str})
    df_chunk = df_chunk[df_chunk['folder'] == folder_id].reset_index(drop=True)
    
    tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, prefix=f'chunk_{folder_id}_')
    df_chunk.to_csv(tmp.name, index=False)
    tmp.close()
    
    ds = GeoguessrHierarchicalDataset(csv_path=tmp.name, image_dir=EXTRA_IMG_DIR, transform=val_transform)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    # Store temp path on loader so we can clean up later
    loader._tmp_csv_path = tmp.name
    return loader
def cleanup_loader(loader):
    """Remove temp CSV after we're done with a loader."""
    if hasattr(loader, '_tmp_csv_path') and os.path.exists(loader._tmp_csv_path):
        os.remove(loader._tmp_csv_path)
def get_augmented_original_loader(aug_transform):
    """Returns the 19k original dataset with a specific augmentation applied."""
    ds = GeoguessrHierarchicalDataset(csv_path=TRAIN_CSV, image_dir=TRAIN_IMG_DIR, transform=aug_transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
# Original clean loader (no augmentation, just resize/normalize)
clean_train_loader = get_augmented_original_loader(val_transform)

Initializing SigLIP Hierarchical Model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  372MB            

model.safetensors: downloading bytes:           |  0.00B            

c:\Users\Yash T\Desktop\geoguessr\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Yash T\.cache\huggingface\hub\models--timm--vit_base_patch16_siglip_256.v2_webli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


The Training (Helper Functions)

In [6]:
# --- TRAINING HELPERS ---
def train_layer_1(loader, epochs=1, lr=LEARNING_RATE_LAYER_1):
    model.freeze_backbone()
    model.unfreeze_layer_1()
    for param in model.coordinate_head.parameters():
        param.requires_grad = False
        
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scaler = GradScaler()
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(loader, desc=f"Layer 1 Epoch {epoch+1}/{epochs}", leave=False):
            imgs = batch['image'].to(device)
            l1, l2, l3 = batch['label_1'].to(device), batch['label_2'].to(device), batch['label_3'].to(device)
            
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                out = model(imgs)
                loss = loss_fn_h1(out['logits_1'], l1) + loss_fn_h2(out['logits_2'], l2) + loss_fn_h3(out['logits_3'], l3)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
        print(f"  Layer 1 Loss: {total_loss/len(loader):.4f}")
def train_layer_2(loader, epochs=1, lr=LEARNING_RATE_LAYER_2):
    model.freeze_backbone()
    model.freeze_layer_1()
    for param in model.coordinate_head.parameters():
        param.requires_grad = True
        
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scaler = GradScaler()
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(loader, desc=f"Layer 2 Epoch {epoch+1}/{epochs}", leave=False):
            imgs = batch['image'].to(device)
            t_lat, t_lon = batch['latitude'].to(device), batch['longitude'].to(device)
            target_xyz = latlon_to_cartesian(t_lat, t_lon)
            
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                out = model(imgs)
                # [BUG 4 FIX] Detach probs so gradients don't leak into frozen heads
                pred_xyz = out['pred_xyz'].detach()
                # Re-run coordinate head with detached probs for clean gradient flow
                coord_input = torch.cat([
                    out['features'].detach(), 
                    out['probs_1'].detach(), 
                    out['probs_2'].detach(), 
                    out['probs_3'].detach()
                ], dim=1)
                raw_xyz = model.coordinate_head(coord_input)
                pred_xyz = F.normalize(raw_xyz, p=2, dim=1)
                loss = loss_fn_coord(pred_xyz, target_xyz)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
        print(f"  Layer 2 (Coord) Loss: {total_loss/len(loader):.4f}")
def train_e2e(loader, epochs=1):
    model.unfreeze_backbone()
    model.unfreeze_layer_1()
    for param in model.coordinate_head.parameters():
        param.requires_grad = True
        
    # Apply the 3 distinct learning rates!
    optimizer = optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': LEARNING_RATE_BACKBONE},
        {'params': model.head_1.parameters(), 'lr': LEARNING_RATE_LAYER_1},
        {'params': model.head_2.parameters(), 'lr': LEARNING_RATE_LAYER_1},
        {'params': model.head_3.parameters(), 'lr': LEARNING_RATE_LAYER_1},
        {'params': model.coordinate_head.parameters(), 'lr': LEARNING_RATE_LAYER_2}
    ])
    
    scaler = GradScaler()
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(loader, desc=f"E2E Epoch {epoch+1}/{epochs}", leave=False):
            imgs = batch['image'].to(device)
            l1, l2, l3 = batch['label_1'].to(device), batch['label_2'].to(device), batch['label_3'].to(device)
            t_lat, t_lon = batch['latitude'].to(device), batch['longitude'].to(device)
            target_xyz = latlon_to_cartesian(t_lat, t_lon)
            
            optimizer.zero_grad(set_to_none=True) # Speed boost!
            with autocast():
                out = model(imgs)
                loss_clf = loss_fn_h1(out['logits_1'], l1) + loss_fn_h2(out['logits_2'], l2) + loss_fn_h3(out['logits_3'], l3)
                loss_reg = loss_fn_coord(out['pred_xyz'], target_xyz)
                loss = loss_clf + loss_reg
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
        print(f"  End-to-End Loss: {total_loss/len(loader):.4f}")

The Evaluation & Save Checkpoint Engine

In [7]:
def evaluate(loader, desc="Evaluating"):
    model.eval()
    errors = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc, leave=False):
            imgs = batch['image'].to(device)
            t_lat, t_lon = batch['latitude'].to(device), batch['longitude'].to(device)
            
            with autocast():
                out = model(imgs)
                dist = haversine_distance(out['pred_lat'], out['pred_lon'], t_lat, t_lon)
                errors.extend(dist.cpu().tolist())
                
    median_error = np.median(errors)
    mean_error = np.mean(errors)
    print(f"  [{desc}] Median Error: {median_error:.2f} km | Mean Error: {mean_error:.2f} km")
    return median_error

CHECKPOINT_DIR = os.path.join(BASE_DIR, 'saved_models')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(step_name):
    """Save model weights after completing a training step."""
    path = os.path.join(CHECKPOINT_DIR, f'hierarchical_{step_name}.pth')
    torch.save(model.state_dict(), path)
    print(f"  💾 Checkpoint saved: {path}")

def load_checkpoint(step_name):
    """Resume from a previously saved checkpoint."""
    path = os.path.join(CHECKPOINT_DIR, f'hierarchical_{step_name}.pth')
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        print(f"  ✅ Resumed from checkpoint: {path}")
        return True
    else:
        print(f"  ⚠️ No checkpoint found at: {path}")
        return False

The Submission Generation Engine

In [8]:
class TestImageDataset(Dataset):
    """Minimal dataset for inference — no labels required."""
    def __init__(self, image_dir, transform):
        self.image_dir = image_dir
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.jpg')])
        
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        fname = self.image_files[idx]
        img_path = os.path.join(self.image_dir, fname)
        
        try:
            import cv2
            cv2.setNumThreads(0) 
            image_np = cv2.imread(img_path)
            if image_np is not None:
                image_np = cv2.cvtColor(image_np, cv2.COLOR_BGR2RGB)
            else:
                raise ValueError("cv2 failed")
        except Exception:
            with Image.open(img_path) as img:
                image_np = np.array(img.convert('RGB'))
        
        augmented = self.transform(image=image_np)
        return {'image': augmented['image'], 'image_id': fname.replace('.jpg', '')}

def generate_submission(filename):
    print(f"Generating submission: {filename}")
    
    test_dataset = TestImageDataset(image_dir=TEST_IMG_DIR, transform=val_transform)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Load GeoJSON for Ocean Snapping
    with open(GEOJSON_PATH) as f:
        countries_data = json.load(f)
    
    # Pre-parse all geometries once for speed
    country_shapes = []
    for feature in countries_data['features']:
        try:
            country_shapes.append(shape(feature['geometry']))
        except Exception:
            continue
    
    def snap_to_land(lat, lon):
        point = Point(lon, lat)
        min_dist = float('inf')
        closest_point = (lat, lon)
        for polygon in country_shapes:
            if polygon.contains(point):
                return lat, lon
            dist = polygon.distance(point)
            if dist < min_dist:
                min_dist = dist
                closest_point = (polygon.centroid.y, polygon.centroid.x)
        return closest_point
    
    results = []
    model.eval()
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Inference (Raw)"):
            imgs = batch['image'].to(device)
            img_ids = batch['image_id']
            
            with autocast():
                out_raw = model(imgs)
            
            raw_lat = out_raw['pred_lat'].cpu().numpy()
            raw_lon = out_raw['pred_lon'].cpu().numpy()
            raw_p1 = out_raw['probs_1'].max(dim=1)[0].cpu().numpy()
            raw_p2 = out_raw['probs_2'].max(dim=1)[0].cpu().numpy()
            raw_p3 = out_raw['probs_3'].max(dim=1)[0].cpu().numpy()
            
            for i in range(len(imgs)):
                img_id = img_ids[i]
                
                # --- Process RAW ---
                r_lat, r_lon = snap_to_land(raw_lat[i], raw_lon[i])
                r1_r = np.clip(14400 * np.exp(-4.6 * raw_p1[i]), 144, 14400)
                r2_r = np.clip(10000 * np.exp(-4.6 * raw_p2[i]), 100, 10000)
                r3_r = np.clip(6400 * np.exp(-4.6 * raw_p3[i]), 64, 6400)
                rad_raw = (4 * r1_r + 36 * r2_r + 216 * r3_r) / 256
                
                results.append({
                    'image_id': f"{img_id}.jpg",
                    'pred_lat': r_lat,
                    'pred_lon': r_lon,
                    'pred_radius_km': rad_raw
                })
                
    pd.DataFrame(results).to_csv(os.path.join('submissions', filename), index=False)
    print(f"  Saved to submissions/{filename}")

The Master 19-Step Curriculum Execution

In [11]:
aug_crop = A.Compose([A.RandomResizedCrop(size=(IMG_H, IMG_W), scale=(0.80, 0.95), p=1.0), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
aug_color = A.Compose([A.Resize(IMG_H, IMG_W), A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02, p=1.0), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
aug_blur = A.Compose([A.Resize(IMG_H, IMG_W), A.GaussianBlur(blur_limit=(2, 6), p=1.0), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
aug_compress = A.Compose([A.Resize(IMG_H, IMG_W), A.ImageCompression(quality_range=(80, 95), p=1.0), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
aug_erase = A.Compose([A.Resize(IMG_H, IMG_W), A.CoarseDropout(num_holes_range=(2, 5), hole_height_range=(1, 10), hole_width_range=(1, 10), p=1.0), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
isolated_augs = [
    ("Random Crop", get_augmented_original_loader(aug_crop)),
    ("Color Jitter", get_augmented_original_loader(aug_color)),
    ("Gaussian Blur", get_augmented_original_loader(aug_blur)),
    ("Image Compression", get_augmented_original_loader(aug_compress)),
    ("Random Erase", get_augmented_original_loader(aug_erase))
]

import random

def run_extra_pass(pass_num, folders):
    # Make a copy of the folders list and shuffle it
    shuffled_folders = list(folders)
    random.shuffle(shuffled_folders)
    
    print(f"\n{'='*55}")
    print(f"Executing PASS {pass_num}")
    print(f"Randomized Folder Order: {shuffled_folders}")
    print(f"{'='*55}")
    
    for folder in shuffled_folders:
        print(f"\n--- Loading Extra Folder {folder} ---")
        loader = get_extra_loader(folder)
        train_layer_1(loader, epochs=1)
        train_layer_2(loader, epochs=1)
        cleanup_loader(loader)
        del loader

In [12]:
# 1. Pass 1
LEARNING_RATE_BACKBONE = 2e-6
LEARNING_RATE_LAYER_1 = 1e-4
LEARNING_RATE_LAYER_2 = 4e-5
run_extra_pass(1, folders=['00', '01', '02', '03'])


Executing PASS 1
Randomized Folder Order: ['03', '02', '00', '01']

--- Loading Extra Folder 03 ---


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Layer 1 Epoch 1/1:   0%|          | 0/2500 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Layer 1 Loss: 7.3660


...

  Layer 1 Loss: 4.6410


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Layer 2 Epoch 1/1:   0%|          | 0/2500 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Layer 2 (Coord) Loss: 2774.1579


In [13]:
# 2. Pass 2
LEARNING_RATE_BACKBONE = 1e-6
LEARNING_RATE_LAYER_1 = 2.5e-5
LEARNING_RATE_LAYER_2 = 1e-5
run_extra_pass(2, folders=['00', '01', '02', '03'])


Executing PASS 2
Randomized Folder Order: ['02', '00', '03', '01']

--- Loading Extra Folder 02 ---


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Layer 1 Epoch 1/1:   0%|          | 0/2500 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Layer 1 Loss: 4.4220


...

  Layer 1 Loss: 4.0621


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Layer 2 Epoch 1/1:   0%|          | 0/2500 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Layer 2 (Coord) Loss: 2502.0959


In [14]:
# 3. End-to-End Fine-Tuning (19k original clean)
LEARNING_RATE_BACKBONE = 2e-6
LEARNING_RATE_LAYER_1 = 5e-5
LEARNING_RATE_LAYER_2 = 2e-5
print("\n[Step 3] E2E Fine-Tuning on Original 19k (5 Epochs)")
train_e2e(clean_train_loader, epochs=5)


[Step 3] E2E Fine-Tuning on Original 19k (5 Epochs)


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:81: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


E2E Epoch 1/5:   0%|          | 0/951 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  End-to-End Loss: 3917.0832


...

  End-to-End Loss: 2696.4777


E2E Epoch 4/5:   0%|          | 0/951 [00:00<?, ?it/s]

  End-to-End Loss: 2310.6928


E2E Epoch 5/5:   0%|          | 0/951 [00:00<?, ?it/s]

  End-to-End Loss: 1993.4708


In [15]:
# 4. Cross-Validation (Folder 04)
print("\n[Step 4] Cross-Validation on Folder 04 Holdout")
loader_04 = get_extra_loader('04')
evaluate(loader_04, desc="Folder 04 Eval")


[Step 4] Cross-Validation on Folder 04 Holdout


Folder 04 Eval:   0%|          | 0/507 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\667953047.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  [Folder 04 Eval] Median Error: 2098.79 km | Mean Error: 3705.99 km


np.float64(2098.790283203125)

In [16]:
# 5. Incorporate Holdout Data
print("\n[Step 5] Incorporating Holdout Data (Folder 04, 2 Epochs E2E)")
LEARNING_RATE_BACKBONE = 2e-6
LEARNING_RATE_LAYER_1 = 1e-4
LEARNING_RATE_LAYER_2 = 4e-5
run_extra_pass(1, folders=['04'])
LEARNING_RATE_BACKBONE = 1e-6
LEARNING_RATE_LAYER_1 = 2.5e-5
LEARNING_RATE_LAYER_2 = 1e-5
run_extra_pass(2, folders=['04'])


[Step 5] Incorporating Holdout Data (Folder 04, 2 Epochs E2E)

Executing PASS 1
Randomized Folder Order: ['04']

--- Loading Extra Folder 04 ---


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Layer 1 Epoch 1/1:   0%|          | 0/507 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Layer 1 Loss: 7.9632


...

  Layer 1 Loss: 7.0057


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Layer 2 Epoch 1/1:   0%|          | 0/507 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Layer 2 (Coord) Loss: 2918.5816


In [17]:
# 6. Polish the model
LEARNING_RATE_BACKBONE = 1e-6
LEARNING_RATE_LAYER_1 = 2.5e-5
LEARNING_RATE_LAYER_2 = 1e-5
print("\n[Step 6] Polish E2E on Original (3 Epochs)")
train_e2e(clean_train_loader, epochs=3)


[Step 6] Polish E2E on Original (3 Epochs)


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:81: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


E2E Epoch 1/3:   0%|          | 0/951 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  End-to-End Loss: 1780.7965


E2E Epoch 2/3:   0%|          | 0/951 [00:00<?, ?it/s]

  End-to-End Loss: 1544.6015


E2E Epoch 3/3:   0%|          | 0/951 [00:00<?, ?it/s]

  End-to-End Loss: 1423.8960


In [18]:
save_checkpoint("phase 1 training")

  💾 Checkpoint saved: C:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\saved_models\hierarchical_phase 1 training.pth


In [19]:
# 7. Evaluate on original
print("\n[Step 7] Evaluating on Original Dataset")
evaluate(clean_train_loader, desc="Original Eval Post-Polish 1")


[Step 7] Evaluating on Original Dataset


Original Eval Post-Polish 1:   0%|          | 0/951 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\667953047.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  [Original Eval Post-Polish 1] Median Error: 722.08 km | Mean Error: 1262.42 km


np.float64(722.07763671875)

In [20]:
# 8. Generate Submission
generate_submission("submission_step8.csv")

Generating submission: submission_step8.csv


Inference (Raw):   0%|          | 0/25 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\1966076519.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Saved to submissions/submission_step8.csv


In [21]:
# 9. Data augmentation (1 epoch each)
LEARNING_RATE_LAYER_1 = 5e-5
LEARNING_RATE_LAYER_2 = 2e-5
print("\n[Step 9] Isolated Augmentation Training")
for aug_name, aug_loader in isolated_augs:
    print(f"-> Training with {aug_name}")
    train_e2e(aug_loader, epochs=1)


[Step 9] Isolated Augmentation Training
-> Training with Random Crop


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:81: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


E2E Epoch 1/1:   0%|          | 0/951 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  End-to-End Loss: 2352.3358
-> Training with Color Jitter


...

  End-to-End Loss: 2624.4516
-> Training with Image Compression


E2E Epoch 1/1:   0%|          | 0/951 [00:00<?, ?it/s]

  End-to-End Loss: 1549.1102
-> Training with Random Erase


E2E Epoch 1/1:   0%|          | 0/951 [00:00<?, ?it/s]

  End-to-End Loss: 1253.4834


In [22]:
# 10. Pass 3
LEARNING_RATE_LAYER_1 = 2.5e-5
LEARNING_RATE_LAYER_2 = 1e-5
run_extra_pass(3, folders=['00', '01', '02', '03', '04'])


Executing PASS 3
Randomized Folder Order: ['04', '03', '00', '01', '02']

--- Loading Extra Folder 04 ---


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Layer 1 Epoch 1/1:   0%|          | 0/507 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Layer 1 Loss: 6.8305


...

  Layer 1 Loss: 4.2535


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Layer 2 Epoch 1/1:   0%|          | 0/2500 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Layer 2 (Coord) Loss: 2745.7719


In [23]:
# 11. Data augmentation (1 epoch each)
print("\n[Step 11] Isolated Augmentation Training")
for aug_name, aug_loader in isolated_augs:
    print(f"-> Training with {aug_name}")
    train_e2e(aug_loader, epochs=1)


[Step 11] Isolated Augmentation Training
-> Training with Random Crop


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:81: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


E2E Epoch 1/1:   0%|          | 0/951 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  End-to-End Loss: 2258.0407
-> Training with Color Jitter


...

  End-to-End Loss: 2278.8659
-> Training with Image Compression


E2E Epoch 1/1:   0%|          | 0/951 [00:00<?, ?it/s]

  End-to-End Loss: 1329.4977
-> Training with Random Erase


E2E Epoch 1/1:   0%|          | 0/951 [00:00<?, ?it/s]

  End-to-End Loss: 1116.4552


In [24]:
# 12. Pass 4
LEARNING_RATE_LAYER_1 = 5e-5
LEARNING_RATE_LAYER_2 = 2e-5
run_extra_pass(4, folders=['00', '01', '02', '03', '04'])


Executing PASS 4
Randomized Folder Order: ['02', '01', '00', '04', '03']

--- Loading Extra Folder 02 ---


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Layer 1 Epoch 1/1:   0%|          | 0/2500 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Layer 1 Loss: 4.4253


...

  Layer 1 Loss: 4.2475


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Layer 2 Epoch 1/1:   0%|          | 0/2500 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Layer 2 (Coord) Loss: 2718.4662


In [25]:
# 13. Polish the model again
print("\n[Step 12] Second Polish E2E on Original (3 Epochs)")
train_e2e(clean_train_loader, epochs=3)


[Step 12] Second Polish E2E on Original (3 Epochs)


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:81: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


E2E Epoch 1/3:   0%|          | 0/951 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  End-to-End Loss: 1165.9668


E2E Epoch 2/3:   0%|          | 0/951 [00:00<?, ?it/s]

  End-to-End Loss: 1015.6708


E2E Epoch 3/3:   0%|          | 0/951 [00:00<?, ?it/s]

  End-to-End Loss: 945.0464


In [26]:
save_checkpoint("phase 2 training")

  💾 Checkpoint saved: C:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\saved_models\hierarchical_phase 2 training.pth


In [27]:
# 14. Evaluate
print("\n[Step 13] Evaluating on Original Dataset")
evaluate(clean_train_loader, desc="Original Eval Post-Polish 2")


[Step 13] Evaluating on Original Dataset


Original Eval Post-Polish 2:   0%|          | 0/951 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\667953047.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  [Original Eval Post-Polish 2] Median Error: 541.46 km | Mean Error: 864.84 km


np.float64(541.4578247070312)

In [28]:
# 15. Generate submission
generate_submission("submission_step15.csv")

Generating submission: submission_step15.csv


Inference (Raw):   0%|          | 0/25 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\1966076519.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Saved to submissions/submission_step15.csv


In [29]:
# 16. Pass 4
LEARNING_RATE_LAYER_1 = 2.5e-5
LEARNING_RATE_LAYER_2 = 1e-5
run_extra_pass(5, folders=['00', '01', '02', '03', '04'])


Executing PASS 5
Randomized Folder Order: ['02', '04', '03', '01', '00']

--- Loading Extra Folder 02 ---


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Layer 1 Epoch 1/1:   0%|          | 0/2500 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Layer 1 Loss: 4.3248


...

  Layer 1 Loss: 4.1785


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Layer 2 Epoch 1/1:   0%|          | 0/2500 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Layer 2 (Coord) Loss: 2654.5873


In [30]:
# 17. Final Polish
print("\n[Step 16] Final Polish E2E on Original (4 Epochs)")
train_e2e(clean_train_loader, epochs=4)


[Step 16] Final Polish E2E on Original (4 Epochs)


C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:81: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


E2E Epoch 1/4:   0%|          | 0/951 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\291821944.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  End-to-End Loss: 1032.2614


...

  End-to-End Loss: 899.6529


E2E Epoch 3/4:   0%|          | 0/951 [00:00<?, ?it/s]

  End-to-End Loss: 842.0651


E2E Epoch 4/4:   0%|          | 0/951 [00:00<?, ?it/s]

  End-to-End Loss: 794.7343


In [31]:
save_checkpoint("phase 3 training")

  💾 Checkpoint saved: C:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\saved_models\hierarchical_phase 3 training.pth


In [32]:
# 18. Final Evaluation
print("\n[Step 17] Final Evaluation on Original Dataset")
evaluate(clean_train_loader, desc="Original Eval Final")


[Step 17] Final Evaluation on Original Dataset


Original Eval Final:   0%|          | 0/951 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\667953047.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  [Original Eval Final] Median Error: 455.92 km | Mean Error: 734.77 km


np.float64(455.92140197753906)

In [33]:
# 19. Generate final submission
generate_submission("submission_final.csv")
print("\n🎉 ALL 19 STEPS COMPLETE 🎉")

Generating submission: submission_final.csv


Inference (Raw):   0%|          | 0/25 [00:00<?, ?it/s]

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\1966076519.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Saved to submissions/submission_final.csv

🎉 ALL 19 STEPS COMPLETE 🎉


In [34]:
import numpy as np

def inspect_submission_probabilities():
    model.eval()
    # Create a small loader that shuffles so you get different images every time you run this cell
    test_dataset = TestImageDataset(image_dir=TEST_IMG_DIR, transform=val_transform)
    inspect_loader = DataLoader(test_dataset, batch_size=4, shuffle=True, num_workers=0) 
    
    batch = next(iter(inspect_loader))
    imgs = batch['image'].to(device)
    img_ids = batch['image_id']
    
    with torch.no_grad():
        with autocast():
            out = model(imgs)
            
    # Extract the full probability distributions
    p1 = out['probs_1'].cpu().numpy()
    p2 = out['probs_2'].cpu().numpy()
    p3 = out['probs_3'].cpu().numpy()
    
    for i in range(len(imgs)):
        print(f"\n{'='*50}")
        print(f"🌍 Image ID: {img_ids[i]}.jpg")
        print(f"{'='*50}")
        
        # --- Head 1 (Macro - 4 classes) ---
        top3_idx1 = np.argsort(p1[i])[::-1][:3] # Get indices of top 3 highest probabilities
        print(f"Head 1 (Macro - 4 Regions):")
        for rank, idx in enumerate(top3_idx1):
            if rank == 0:
                print(f"  🏆 Top Choice : Class {idx:<3d} -> {p1[i][idx]*100:>5.1f}% confidence")
            else:
                print(f"  #{rank+1} Runner Up: Class {idx:<3d} -> {p1[i][idx]*100:>5.1f}% confidence")
                
        # --- Head 2 (Meso - 36 classes) ---
        top3_idx2 = np.argsort(p2[i])[::-1][:3]
        print(f"\nHead 2 (Meso - 36 Regions):")
        for rank, idx in enumerate(top3_idx2):
            if rank == 0:
                print(f"  🏆 Top Choice : Class {idx:<3d} -> {p2[i][idx]*100:>5.1f}% confidence")
            else:
                print(f"  #{rank+1} Runner Up: Class {idx:<3d} -> {p2[i][idx]*100:>5.1f}% confidence")
                
        # --- Head 3 (Micro - 216 classes) ---
        top3_idx3 = np.argsort(p3[i])[::-1][:3]
        print(f"\nHead 3 (Micro - 216 Regions):")
        for rank, idx in enumerate(top3_idx3):
            if rank == 0:
                print(f"  🏆 Top Choice : Class {idx:<3d} -> {p3[i][idx]*100:>5.1f}% confidence")
            else:
                print(f"  #{rank+1} Runner Up: Class {idx:<3d} -> {p3[i][idx]*100:>5.1f}% confidence")

# Run the inspection
inspect_submission_probabilities()


🌍 Image ID: 568e76881ea1f0d1.jpg
Head 1 (Macro - 4 Regions):
  🏆 Top Choice : Class 1   ->  86.7% confidence
  #2 Runner Up: Class 2   ->  10.5% confidence
  #3 Runner Up: Class 0   ->   2.5% confidence

Head 2 (Meso - 36 Regions):
  🏆 Top Choice : Class 20  ->  47.7% confidence
  #2 Runner Up: Class 0   ->  28.9% confidence
  #3 Runner Up: Class 15  ->   3.3% confidence

Head 3 (Micro - 216 Regions):
  🏆 Top Choice : Class 121 ->  22.2% confidence
  #2 Runner Up: Class 93  ->  13.0% confidence
  #3 Runner Up: Class 154 ->   6.8% confidence

🌍 Image ID: faea072d54cddb75.jpg
Head 1 (Macro - 4 Regions):
  🏆 Top Choice : Class 0   ->  97.1% confidence
  #2 Runner Up: Class 2   ->   2.2% confidence
  #3 Runner Up: Class 1   ->   0.6% confidence

Head 2 (Meso - 36 Regions):
  🏆 Top Choice : Class 1   ->  62.5% confidence
  #2 Runner Up: Class 28  ->  18.6% confidence
  #3 Runner Up: Class 31  ->   7.0% confidence

Head 3 (Micro - 216 Regions):
  🏆 Top Choice : Class 62  ->  21.7% confidenc

C:\Users\Yash T\AppData\Local\Temp\ipykernel_19484\85045475.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


In [12]:
# DIAGNOSTIC: Compare SigLIP vs ConvNext predictions on the test set
import pandas as pd

sub_new = pd.read_csv('submissions/submission_final.csv')
sub_old = pd.read_csv('submissions/final_submission_convnextv2_base.csv')

# Load GeoJSON for country checking
with open(GEOJSON_PATH) as f:
    countries_data = json.load(f)
country_shapes_named = []
for feature in countries_data['features']:
    try:
        country_shapes_named.append((feature['properties'].get('ISO_A2', 'XX'), shape(feature['geometry'])))
    except:
        continue

def get_country(lat, lon):
    point = Point(lon, lat)
    for name, polygon in country_shapes_named:
        if polygon.contains(point):
            return name
    return "OCEAN"

merged = pd.merge(sub_new, sub_old, on='image_id', suffixes=('_siglip', '_convnext'))

# Check how many predictions land in the ocean
siglip_ocean = 0
convnext_ocean = 0
siglip_countries = []
convnext_countries = []

for _, row in merged.iterrows():
    sc = get_country(row.pred_lat_siglip, row.pred_lon_siglip)
    cc = get_country(row.pred_lat_convnext, row.pred_lon_convnext)
    siglip_countries.append(sc)
    convnext_countries.append(cc)
    if sc == "OCEAN": siglip_ocean += 1
    if cc == "OCEAN": convnext_ocean += 1

print(f"=== OCEAN PREDICTIONS ===")
print(f"SigLIP  in ocean: {siglip_ocean} / 500")
print(f"ConvNext in ocean: {convnext_ocean} / 500")

print(f"\n=== COORDINATE SPREAD ===")
print(f"SigLIP  lat std: {sub_new.pred_lat.std():.1f}, lon std: {sub_new.pred_lon.std():.1f}")
print(f"ConvNext lat std: {sub_old.pred_lat.std():.1f}, lon std: {sub_old.pred_lon.std():.1f}")

print(f"\n=== RADIUS COMPARISON ===")
print(f"SigLIP  radius: min={sub_new.pred_radius_km.min():.0f}, median={sub_new.pred_radius_km.median():.0f}, max={sub_new.pred_radius_km.max():.0f}")
print(f"ConvNext radius: min={sub_old.pred_radius_km.min():.0f}, median={sub_old.pred_radius_km.median():.0f}, max={sub_old.pred_radius_km.max():.0f}")

# Check how many predictions differ by more than 5000 km
from scripts.losses_hierarchical import haversine_distance
import torch

lat1 = torch.tensor(merged.pred_lat_siglip.values)
lon1 = torch.tensor(merged.pred_lon_siglip.values)
lat2 = torch.tensor(merged.pred_lat_convnext.values)
lon2 = torch.tensor(merged.pred_lon_convnext.values)
dists = haversine_distance(lat1, lon1, lat2, lon2).numpy()

print(f"\n=== PREDICTION DISAGREEMENT (SigLIP vs ConvNext) ===")
print(f"Median distance between predictions: {np.median(dists):.0f} km")
print(f"Mean distance between predictions: {np.mean(dists):.0f} km")
print(f"Predictions > 5000 km apart: {(dists > 5000).sum()} / 500")
print(f"Predictions > 3000 km apart: {(dists > 3000).sum()} / 500")
print(f"Predictions < 1000 km apart: {(dists < 1000).sum()} / 500")

=== OCEAN PREDICTIONS ===
SigLIP  in ocean: 41 / 500
ConvNext in ocean: 186 / 500

=== COORDINATE SPREAD ===
SigLIP  lat std: 30.7, lon std: 66.4
ConvNext lat std: 27.5, lon std: 68.8

=== RADIUS COMPARISON ===
SigLIP  radius: min=98, median=1826, max=4444
ConvNext radius: min=64, median=6385, max=6400

=== PREDICTION DISAGREEMENT (SigLIP vs ConvNext) ===
Median distance between predictions: 2368 km
Mean distance between predictions: 3900 km
Predictions > 5000 km apart: 125 / 500
Predictions > 3000 km apart: 206 / 500
Predictions < 1000 km apart: 105 / 500


In [6]:
# A/B TEST: SigLIP coordinates + safe radius
import pandas as pd

sub = pd.read_csv('submissions/submission_final.csv')
sub['pred_radius_km'] = 6000.0  # Override ALL radii to be safe
sub.to_csv('submissions/submission_AB_test_safe_radius.csv', index=False)
print("Saved! Submit this to Kaggle to isolate the radius effect.")

Saved! Submit this to Kaggle to isolate the radius effect.


In [ ]:
import pandas as pd, numpy as np, torch, torch.nn.functional as F

sub_sig = pd.read_csv('submissions/submission_final.csv')
sub_cnx = pd.read_csv('submissions/final_submission_convnextv2_base.csv')

merged = pd.merge(sub_sig, sub_cnx, on='image_id', suffixes=('_sig', '_cnx'))

results = []
for _, row in merged.iterrows():
    # Convert both predictions to unit sphere
    lat1, lon1 = np.radians(row.pred_lat_sig), np.radians(row.pred_lon_sig)
    lat2, lon2 = np.radians(row.pred_lat_cnx), np.radians(row.pred_lon_cnx)
    
    xyz1 = np.array([np.cos(lat1)*np.cos(lon1), np.cos(lat1)*np.sin(lon1), np.sin(lat1)])
    xyz2 = np.array([np.cos(lat2)*np.cos(lon2), np.cos(lat2)*np.sin(lon2), np.sin(lat2)])
    
    # Weighted average: trust SigLIP slightly more (better country prediction)
    w_sig, w_cnx = 0.5, 0.5
    avg_xyz = w_sig * xyz1 + w_cnx * xyz2
    avg_xyz = avg_xyz / np.linalg.norm(avg_xyz)
    
    pred_lat = np.degrees(np.arcsin(np.clip(avg_xyz[2], -1, 1)))
    pred_lon = np.degrees(np.arctan2(avg_xyz[1], avg_xyz[0]))
    
    # Conservative radius
    radius = max(row.pred_radius_km_sig, row.pred_radius_km_cnx)
    
    results.append({
        'image_id': row.image_id,
        'pred_lat': pred_lat,
        'pred_lon': pred_lon,
        'pred_radius_km': radius
    })

pd.DataFrame(results).to_csv('submissions/submission_ensemble.csv', index=False)
print("Saved ensemble submission!")

Saved ensemble submission!


In [8]:
import pandas as pd, numpy as np
from shapely.geometry import Point, shape
import json

sub_sig = pd.read_csv('submissions/submission_final.csv')
sub_cnx = pd.read_csv('submissions/final_submission_convnextv2_base.csv')

with open(GEOJSON_PATH) as f:
    countries_data = json.load(f)
country_polys = []
for feature in countries_data['features']:
    try:
        country_polys.append(shape(feature['geometry']))
    except:
        continue

def is_on_land(lat, lon):
    point = Point(lon, lat)
    for poly in country_polys:
        if poly.contains(point):
            return True
    return False

merged = pd.merge(sub_sig, sub_cnx, on='image_id', suffixes=('_sig', '_cnx'))

results = []
swapped = 0
for _, row in merged.iterrows():
    cnx_on_land = is_on_land(row.pred_lat_cnx, row.pred_lon_cnx)
    sig_on_land = is_on_land(row.pred_lat_sig, row.pred_lon_sig)
    
    if cnx_on_land:
        # ConvNext is on land → trust it (it scored 50)
        results.append({
            'image_id': row.image_id,
            'pred_lat': row.pred_lat_cnx,
            'pred_lon': row.pred_lon_cnx,
            'pred_radius_km': row.pred_radius_km_cnx
        })
    elif sig_on_land:
        # ConvNext is in ocean but SigLIP is on land → use SigLIP
        swapped += 1
        results.append({
            'image_id': row.image_id,
            'pred_lat': row.pred_lat_sig,
            'pred_lon': row.pred_lon_sig,
            'pred_radius_km': 6000.0  # Safe radius for swapped predictions
        })
    else:
        # Both in ocean → use SigLIP (fewer ocean predictions overall)
        swapped += 1
        results.append({
            'image_id': row.image_id,
            'pred_lat': row.pred_lat_sig,
            'pred_lon': row.pred_lon_sig,
            'pred_radius_km': 6000.0
        })

print(f"Used ConvNext for {500-swapped}/500, swapped to SigLIP for {swapped}/500")
pd.DataFrame(results).to_csv('submissions/submission_smart_ensemble.csv', index=False)
print("Saved! Submit this.")

Used ConvNext for 314/500, swapped to SigLIP for 186/500
Saved! Submit this.


In [9]:
import pandas as pd
from shapely.geometry import Point, shape
from shapely.ops import nearest_points
import json

sub_cnx = pd.read_csv('submissions/final_submission_convnextv2_base.csv')

with open(GEOJSON_PATH) as f:
    countries_data = json.load(f)
country_polys = []
for feature in countries_data['features']:
    try:
        country_polys.append(shape(feature['geometry']))
    except:
        continue

def snap_to_nearest_land(lat, lon):
    """Snaps ocean points to the nearest point on land (not centroid)."""
    point = Point(lon, lat)
    min_dist = float('inf')
    nearest_lat, nearest_lon = lat, lon
    for poly in country_polys:
        if poly.contains(point):
            return lat, lon  # Already on land
        dist = poly.distance(point)
        if dist < min_dist:
            min_dist = dist
            # Get the actual nearest point on the polygon boundary
            near_pt = nearest_points(point, poly)[1]
            nearest_lat, nearest_lon = near_pt.y, near_pt.x
    return nearest_lat, nearest_lon

results = []
snapped = 0
for _, row in sub_cnx.iterrows():
    new_lat, new_lon = snap_to_nearest_land(row.pred_lat, row.pred_lon)
    if new_lat != row.pred_lat or new_lon != row.pred_lon:
        snapped += 1
    results.append({
        'image_id': row.image_id,
        'pred_lat': new_lat,
        'pred_lon': new_lon,
        'pred_radius_km': row.pred_radius_km
    })

print(f"Snapped {snapped}/500 ocean predictions to nearest coastline")
pd.DataFrame(results).to_csv('submissions/submission_convnext_snapped.csv', index=False)
print("Saved! Submit this.")

Snapped 186/500 ocean predictions to nearest coastline
Saved! Submit this.
